# Segmentación por **Watershed** con validación cruzada de **K = 5** y test fijo

Aplica **watershed con marcadores** a todas las imágenes de
`DATASET_FINAL1.mat`.

* validación cruzada **estratificada por tipo de lesión**, K = 5 pliegues
* las imágenes de índice **23–27** (con imagen registrada) son **siempre test** y nunca entran en entrenamiento ni en validación
* el resto de imágenes rota: en cada pliegue, 80 % train+val (reparto interno 80/20) y
  20 % test, que se suma al test fijo
* métricas reportadas como **media** entre pliegues, desglosadas en test total,
  solo rotatorias y solo fijas

Watershed no aprende parámetros: lo que se selecciona en cada pliegue es la
**configuración de marcadores** (mapa de lesión, forma de sembrar los marcadores y
umbrales), igual que en el cuaderno de K-means se seleccionaba `(espacio, k, criterio)`.


### 1. Montar Google Drive

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 2. Parámetros

In [18]:
RUTA_MAT      = "/C:/Users/josem/Desktop/DATASET_FINAL1.pdf

UMBRAL_DICE   = 0.9      # solo se muestran las imágenes que superen este Dice
MAX_MOSTRAR   = 5        # tope de filas en la figura (None = todas)
POSTPROCESO   = True     # limpieza morfológica + mayor componente conexa
RELLENAR      = True     # rellena los huecos negros dentro de la lesión
GUARDAR       = True     # guardar la figura en CARPETA_SALIDA

REDIMENSIONAR = True     # 256x256, como el resto del pipeline del TFG.
IMG_SIZE      = 256      # Ponlo a False para trabajar a resolución nativa

# --- validación cruzada ---
N_SPLITS      = 5        # K = 5 pliegues
VAL_FRACTION  = 0.20     # 20 % del 80 % restante = 16 % del total
RANDOM_STATE  = 42       # semilla fija -> pliegues reproducibles

# --- test fijo -------------------------------------------------------------
# Imágenes con imagen registrada del dataset: SIEMPRE son test, en todos los
# pliegues, y NUNCA entran en entrenamiento ni en validación.
INDICES_TEST_FIJOS = list(range(23, 28))   # 23, 24, 25, 26, 27

# --- watershed -------------------------------------------------------------
SUAVIZADO     = 5        # sigma/tamaño del filtro previo (reduce sobresegmentación)
USAR_BILATERAL = True    # filtro bilateral antes del gradiente: alisa sin perder bordes
PESO_CENTRO   = 0.30     # peso del prior de centralidad en los mapas "*_c"
AREA_MAXIMA   = 0.90     # se descartan regiones que ocupan más del 90 % de la imagen

# Espacio de búsqueda de la configuración: (mapa, siembra, p_fg, p_bg).
#   mapa    -> mapa escalar de "lesionalidad" del que salen los marcadores
#   siembra -> cómo se colocan los marcadores de objeto y de fondo
#   p_fg    -> percentil que define el marcador de OBJETO (más alto = semilla más pequeña)
#   p_bg    -> percentil que define el marcador de FONDO  (más bajo  = fondo más seguro)
# (en las siembras "otsu" los percentiles se ignoran)
CONFIGS_CANDIDATAS = [
    ("a_lab",      "percentil", 90, 40),   # crominancia a*: los hemangiomas son rojizos
    ("a_lab",      "percentil", 85, 50),
    ("a_lab",      "otsu",       0,  0),   # umbral automático + transformada de distancia
    ("a_lab",      "distancia", 85, 40),
    ("a_lab_c",    "percentil", 88, 45),   # color + prior de centralidad
    ("a_menos_b",  "percentil", 90, 40),   # a* - b*: realza el rojo frente al amarillo
    ("a_menos_b",  "otsu",       0,  0),
    ("saturacion", "percentil", 90, 40),   # S de HSV
    ("saturacion", "otsu",       0,  0),
    ("rojez",      "percentil", 88, 45),   # R - (G+B)/2
    ("rojez_c",    "distancia", 85, 40),
    ("oscuridad",  "percentil", 90, 40),   # lesión más oscura que la piel
]

### 3. Importaciones

In [19]:
import os
import h5py
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_fill_holes
from sklearn.model_selection import StratifiedKFold, train_test_split
from collections import Counter

np.random.seed(RANDOM_STATE)
cv2.setRNGSeed(RANDOM_STATE)

### 4. Lectura del dataset

Se cargan **todas** las entradas (no una selección), porque la validación cruzada necesita el
conjunto completo.

In [20]:
f  = h5py.File(RUTA_MAT, "r")
DS = f["DATASET_UNIDO"]

def leer_entrada(i):
    c = [f[r] for r in f[DS[i, 0]][()].ravel()]
    nombre = "".join(chr(x) for x in c[0][()].ravel())
    tipo   = "".join(chr(x) for x in c[2][()].ravel())
    mask   = (c[1][()].T > 0).astype(np.uint8)                          # (H, W) binaria
    img    = (np.transpose(c[3][()], (2, 1, 0)) * 255).clip(0, 255).astype(np.uint8)
    return nombre, tipo, img, mask


N = DS.shape[0]
nombres, tipos, imagenes, mascaras = [], [], [], []

for i in range(N):
    nombre, tipo, img, gt = leer_entrada(i)
    if REDIMENSIONAR:
        # área para la imagen, vecino más próximo para la máscara,
        # que así conserva su carácter estrictamente binario
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        gt  = cv2.resize(gt,  (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    nombres.append(nombre)
    tipos.append(tipo)
    imagenes.append(img)
    mascaras.append((gt > 0).astype(np.uint8))

etiquetas = np.array(tipos)
print(f"{N} imágenes cargadas. ")

130 imágenes cargadas. 


### 5. Segmentación por watershed con marcadores

El watershed interpreta la imagen como un relieve y "inunda" las cuencas desde unos
marcadores. Aplicado en crudo sobre el gradiente produce **sobresegmentación** (cientos de
cuencas por culpa del ruido y del vello), así que se usa la variante **con marcadores**, en
cuatro piezas:

1. **Mapa de lesionalidad** — un escalar por píxel que mide cuánto "parece lesión":
   `a_lab` (canal *a* de LAB), `a_menos_b`, `saturacion`, `rojez` o `oscuridad`. Los mapas
   terminados en `_c` mezclan además un **prior de centralidad** con peso `PESO_CENTRO`,
   el equivalente a lo que hacía `PESO_XY` en K-means. El mapa se normaliza entre sus
   percentiles 1 y 99 para que los umbrales por percentil signifiquen lo mismo en todas
   las imágenes.
2. **Siembra de marcadores** — de ese mapa salen dos máscaras: el marcador de **objeto**
   (percentil `p_fg` alto, o umbral de Otsu seguido de la transformada de distancia) y el
   marcador de **fondo** (percentil `p_bg` bajo, o la dilatación del binario). Lo que queda
   entre ambos es la zona **desconocida** que el watershed tiene que decidir.
3. **Inundación** — `cv2.watershed` sobre la imagen filtrada (bilateral, que alisa la piel
   sin difuminar el borde de la lesión), partiendo de esos marcadores.
4. **Selección de la región de lesión** — igual que en K-means, hay que decir cuál de las
   cuencas es la lesión: se elige la de mayor media del mapa de lesionalidad, descartando
   las que ocupan más del `AREA_MAXIMA` de la imagen (casi siempre piel o fondo).


In [21]:
EPS = 1e-8

def _centralidad(H, W):
    """Mapa 1 en el centro, 0 en el borde. Prior suave de posición."""
    yy, xx = np.mgrid[0:H, 0:W]
    d = np.sqrt(((xx - (W - 1) / 2) / (W / 2)) ** 2 +
                ((yy - (H - 1) / 2) / (H / 2)) ** 2)
    return (1.0 - np.clip(d, 0, 1)).astype(np.float32)


def _norm01(m):
    """Normaliza entre percentiles 1 y 99: robusto frente a brillos y valores extremos."""
    m = m.astype(np.float32)
    lo, hi = float(np.percentile(m, 1)), float(np.percentile(m, 99))
    return np.clip((m - lo) / (hi - lo + EPS), 0, 1).astype(np.float32)


def imagen_filtrada(img):
    """Filtrado previo. El bilateral alisa la piel y el ruido del sensor pero
    respeta el borde de la lesión, que es justo donde debe caer la línea divisoria."""
    if USAR_BILATERAL:
        return cv2.bilateralFilter(img, 9, 50, 50)
    return cv2.GaussianBlur(img, (SUAVIZADO, SUAVIZADO), 0)


def mapa_lesion(img, mapa):
    """Mapa escalar (H, W) en [0, 1] con la 'lesionalidad' de cada píxel."""
    usar_centro = mapa.endswith("_c")
    base = mapa[:-2] if usar_centro else mapa

    suave = cv2.GaussianBlur(img, (SUAVIZADO, SUAVIZADO), 0)
    lab   = cv2.cvtColor(suave, cv2.COLOR_RGB2LAB).astype(np.float32)

    if base == "a_lab":
        m = lab[:, :, 1]
    elif base == "a_menos_b":
        m = lab[:, :, 1] - lab[:, :, 2]
    elif base == "saturacion":
        m = cv2.cvtColor(suave, cv2.COLOR_RGB2HSV)[:, :, 1].astype(np.float32)
    elif base == "oscuridad":
        m = -cv2.cvtColor(suave, cv2.COLOR_RGB2GRAY).astype(np.float32)
    elif base == "rojez":
        r = suave[:, :, 0].astype(np.float32)
        g = suave[:, :, 1].astype(np.float32)
        b = suave[:, :, 2].astype(np.float32)
        m = r - 0.5 * (g + b)
    else:
        raise ValueError(f"mapa desconocido: {mapa}")

    m = _norm01(m)
    if usar_centro:
        H, W = m.shape
        m = (1 - PESO_CENTRO) * m + PESO_CENTRO * _centralidad(H, W)
    return m.astype(np.float32)


def mayor_componente(binaria):
    n, etq = cv2.connectedComponents(binaria.astype(np.uint8))
    if n <= 1:
        return binaria.astype(np.uint8)
    areas = [(etq == j).sum() for j in range(1, n)]
    return (etq == 1 + int(np.argmax(areas))).astype(np.uint8)


def marcadores(m01, siembra, p_fg=90, p_bg=40):
    """Construye la imagen de marcadores que necesita cv2.watershed.

    Convenio de OpenCV: 0 = zona desconocida (a decidir), 1 = fondo,
    >= 2 = un marcador de objeto por componente conexa."""
    k5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    if siembra == "percentil":
        # Semillas directas: lo más 'lesión' es objeto, lo menos es fondo.
        fg = (m01 >= np.percentile(m01, p_fg)).astype(np.uint8)
        bg = (m01 <= np.percentile(m01, p_bg)).astype(np.uint8)

    elif siembra in ("otsu", "distancia"):
        # Binarizado (automático o por percentil) + transformada de distancia:
        # la semilla de objeto es el 'corazón' de la mancha, lejos de su borde.
        u8 = np.uint8(m01 * 255)
        if siembra == "otsu":
            _, binar = cv2.threshold(u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        else:
            _, binar = cv2.threshold(u8, float(np.percentile(u8, p_fg)), 255, cv2.THRESH_BINARY)
        binar = cv2.morphologyEx((binar > 0).astype(np.uint8), cv2.MORPH_OPEN, k5)
        if binar.sum() == 0:
            binar = (m01 >= np.percentile(m01, 95)).astype(np.uint8)
        d  = cv2.distanceTransform(binar, cv2.DIST_L2, 5)
        fg = (d >= 0.5 * d.max()).astype(np.uint8) if d.max() > 0 else binar
        bg = (cv2.dilate(binar, k5, iterations=3) == 0).astype(np.uint8)

    else:
        raise ValueError(f"siembra desconocida: {siembra}")

    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, k5)
    if fg.sum() == 0:                                    # red de seguridad
        fg = (m01 >= np.percentile(m01, 99)).astype(np.uint8)
    if bg.sum() == 0:
        bg = (m01 <= np.percentile(m01, 10)).astype(np.uint8)
    bg[fg > 0] = 0                                       # objeto y fondo no se solapan

    n, mk = cv2.connectedComponents(fg)
    mk = mk.astype(np.int32) + 1                         # fondo del etiquetado -> 1
    mk[(fg == 0) & (bg == 0)] = 0                        # zona desconocida
    return mk, fg, bg


def segmentar_watershed(img, mapa="a_lab", siembra="percentil", p_fg=90, p_bg=40,
                        postproceso=True, rellenar=True):
    """Watershed con marcadores + selección de la cuenca que corresponde a la lesión."""
    m01       = mapa_lesion(img, mapa)
    mk, fg, _ = marcadores(m01, siembra, p_fg, p_bg)

    base = cv2.cvtColor(imagen_filtrada(img), cv2.COLOR_RGB2BGR)   # OpenCV trabaja en BGR
    etq  = cv2.watershed(np.ascontiguousarray(base), mk.copy())    # -1 = línea divisoria

    ids = [j for j in np.unique(etq) if j >= 2]        # las cuencas de objeto
    if not ids:
        seg = (fg > 0).astype(np.uint8)
    else:
        areas  = np.array([(etq == j).mean()      for j in ids])
        puntos = np.array([m01[etq == j].mean()   for j in ids])
        validos = areas < AREA_MAXIMA              # fuera piel y fondo
        if validos.any():
            puntos = np.where(validos, puntos, -np.inf)
        seg = (etq == ids[int(np.argmax(puntos))]).astype(np.uint8)

    if postproceso:
        kk  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        seg = cv2.morphologyEx(seg, cv2.MORPH_OPEN,  kk)
        seg = cv2.morphologyEx(seg, cv2.MORPH_CLOSE, kk)
        if seg.sum() > 0:
            seg = mayor_componente(seg)            # conserva la mayor componente conexa

    if rellenar and seg.sum() > 0:
        seg = binary_fill_holes(seg).astype(np.uint8)   # tapa huecos interiores

    return seg.astype(np.uint8)


# La misma imagen se segmenta muchas veces con la misma configuración (una por
# pliegue y por candidata), así que se memoriza el resultado. Es determinista y
# depende solo de (imagen, configuración): no introduce ninguna fuga de datos.
_CACHE_SEG = {}

def segmentar_cache(i, mapa, siembra, p_fg, p_bg):
    clave = (i, mapa, siembra, p_fg, p_bg, POSTPROCESO, RELLENAR)
    if clave not in _CACHE_SEG:
        _CACHE_SEG[clave] = segmentar_watershed(imagenes[i], mapa, siembra, p_fg, p_bg,
                                                POSTPROCESO, RELLENAR)
    return _CACHE_SEG[clave]

### 6. Métricas y dibujo de contornos

In [25]:
def matriz_confusion(pred, gt):
    """Recuento de VP, VN, FP y FN a nivel de píxel."""
    p, g = pred > 0, gt > 0
    return dict(vp=int(( p &  g).sum()), vn=int((~p & ~g).sum()),
                fp=int(( p & ~g).sum()), fn=int((~p &  g).sum()))

def suma_confusion(a, b):
    return {k: a[k] + b[k] for k in a}

def dice_de_confusion(c):
    return (2 * c["vp"] + EPS) / (2 * c["vp"] + c["fp"] + c["fn"] + EPS)

def metricas_de_confusion(c):
    return {
        "exactitud":     (c["vp"] + c["vn"]) / (c["vp"] + c["vn"] + c["fp"] + c["fn"] + EPS),
        "precision":     (c["vp"] + EPS) / (c["vp"] + c["fp"] + EPS),
        "sensibilidad":  (c["vp"] + EPS) / (c["vp"] + c["fn"] + EPS),
        "especificidad": (c["vn"] + EPS) / (c["vn"] + c["fp"] + EPS),
    }

def dice(a, b):
    return dice_de_confusion(matriz_confusion(a, b))


def evaluar(indices, mapa, siembra, p_fg, p_bg):
    """Evalúa una configuración de watershed sobre un subconjunto de índices.

    Distingue las dos formas de agregar, tal y como se explica en §6.2 de la memoria:
      * Dice -> se calcula por imagen y se promedia (macro). Cada lesión pesa igual.
      * F1   -> se acumulan VP, FP y FN de todas las imágenes y se calcula una sola
                vez sobre el total (micro). Cada píxel pesa igual, así que las
                lesiones grandes influyen más.
    Por eso Dice y F1 comparten fórmula pero no valor."""
    dices, total = [], dict(vp=0, vn=0, fp=0, fn=0)
    for i in indices:
        seg = segmentar_cache(i, mapa, siembra, p_fg, p_bg)
        c = matriz_confusion(seg, mascaras[i])
        dices.append(dice_de_confusion(c))
        total = suma_confusion(total, c)
    res = metricas_de_confusion(total)
    res["dice"] = float(np.mean(dices))     # macro, por imagen
    res["f1"]   = dice_de_confusion(total)  # micro, global
    return res


def elegir_configuracion(indices_ajuste):
    """Elige la configuración (mapa, siembra, p_fg, p_bg) que maximiza el Dice en
    entrenamiento + validación. Ninguna imagen de test interviene en esta decisión."""
    mejor, mejor_dice = None, -1.0
    for mapa, siembra, p_fg, p_bg in CONFIGS_CANDIDATAS:
        d = evaluar(indices_ajuste, mapa, siembra, p_fg, p_bg)["dice"]
        if d > mejor_dice:
            mejor, mejor_dice = (mapa, siembra, p_fg, p_bg), d
    return mejor, mejor_dice


def dibujar_contornos(img, seg, gt, grosor=None):
    o = img.copy()
    if grosor is None:
        grosor = max(2, int(round(max(img.shape[:2]) / 300)))   # grosor según tamaño
    cont_gt,  _ = cv2.findContours(gt.astype(np.uint8),  cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cont_seg, _ = cv2.findContours(seg.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(o, cont_gt,  -1, (255, 0, 0), grosor)   # ground truth -> rojo
    cv2.drawContours(o, cont_seg, -1, (0, 255, 0), grosor)   # watershed    -> verde
    return o

### 7. Validación cruzada estratificada de 5 pliegues con test fijo

Las imágenes de índice **23–27** (las que tienen imagen registrada) se reservan como
**test en todos los pliegues** y **nunca** participan en la elección de la configuración
de watershed (ni en entrenamiento ni en validación).

El resto del dataset se reparte con `StratifiedKFold`, de modo que en cada pliegue:

* **test** = pliegue rotatorio + las 5 imágenes fijas
* **train / val** = el 80 % restante de las imágenes rotatorias (80/20 interno)

Como las fijas se evalúan K veces (una por pliegue, con la configuración de cada uno),
su Dice fuera de pliegue se reporta como **media entre los K pliegues**.

In [26]:
# --- conjunto de test fijo -------------------------------------------------
FIJOS = np.array(sorted(set(INDICES_TEST_FIJOS)), dtype=int)
assert FIJOS.min() >= 0 and FIJOS.max() < N, "INDICES_TEST_FIJOS fuera de rango"

# El resto del dataset es lo único que rota entre entrenamiento, validación y test
RESTO = np.array([i for i in range(N) if i not in set(FIJOS.tolist())], dtype=int)

print(f"Test fijo ({len(FIJOS)} imágenes, siempre en test, nunca en train/val):")
for i in FIJOS:
    print(f"   idx {i:>3}  {nombres[i]}")
print(f"Imágenes que rotan en la validación cruzada: {len(RESTO)}\n")

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

resultados_pliegue       = []   # test completo = pliegue rotatorio + fijas
resultados_pliegue_rot   = []   # solo la parte rotatoria del test
resultados_pliegue_fijos = []   # solo las imágenes fijas
configs_elegidas         = []

# predicción fuera de pliegue para cada imagen
seg_oof    = [None] * N
dice_oof   = np.zeros(N)
pliegue_de = np.zeros(N, dtype=int)
dice_fijos_por_pliegue = {i: [] for i in FIJOS}   # las fijas se predicen K veces

print(f"Validación cruzada estratificada de {N_SPLITS} pliegues — watershed\n")

for pliegue, (pos_trval, pos_test) in enumerate(
        skf.split(np.zeros(len(RESTO)), etiquetas[RESTO]), 1):

    idx_trval    = RESTO[pos_trval]
    idx_test_rot = RESTO[pos_test]
    # El test de cada pliegue = parte rotatoria + las imágenes fijas
    idx_test     = np.concatenate([idx_test_rot, FIJOS])

    # Separación interna entrenamiento / validación
    try:
        idx_train, idx_val = train_test_split(
            idx_trval, test_size=VAL_FRACTION,
            stratify=etiquetas[idx_trval], random_state=RANDOM_STATE)
    except ValueError:
        # alguna clase tiene muy pocas muestras para estratificar el subreparto
        idx_train, idx_val = train_test_split(
            idx_trval, test_size=VAL_FRACTION, random_state=RANDOM_STATE)

    # La configuración se decide sobre entrenamiento + validación...
    (mapa, siembra, p_fg, p_bg), dice_ajuste = elegir_configuracion(
        np.concatenate([idx_train, idx_val]))
    # ...y se aplica al pliegue de test, no visto en la selección.
    res       = evaluar(idx_test,     mapa, siembra, p_fg, p_bg)
    res_rot   = evaluar(idx_test_rot, mapa, siembra, p_fg, p_bg)
    res_fijos = evaluar(FIJOS,        mapa, siembra, p_fg, p_bg)

    for i in idx_test:
        s = segmentar_cache(i, mapa, siembra, p_fg, p_bg)
        d = dice(s, mascaras[i])
        seg_oof[i]    = s          # para las fijas queda la del último pliegue
        dice_oof[i]   = d
        pliegue_de[i] = pliegue
        if i in dice_fijos_por_pliegue:
            dice_fijos_por_pliegue[i].append(d)

    resultados_pliegue.append(res)
    resultados_pliegue_rot.append(res_rot)
    resultados_pliegue_fijos.append(res_fijos)
    configs_elegidas.append((mapa, siembra, p_fg, p_bg))



# El Dice fuera de pliegue de las imágenes fijas se promedia entre los K pliegues
for i in FIJOS:
    dice_oof[i] = float(np.mean(dice_fijos_por_pliegue[i]))

Test fijo (5 imágenes, siempre en test, nunca en train/val):
   idx  23  imagen20.jpg
   idx  24  imagen21.jpg
   idx  25  imagen24.jpg
   idx  26  imagen29.jpg
   idx  27  imagen3.jpg
Imágenes que rotan en la validación cruzada: 125

Validación cruzada estratificada de 5 pliegues — watershed



### 8. Resumen

In [27]:
CLAVES  = ["exactitud", "sensibilidad", "especificidad", "precision", "dice", "f1"]
NOMBRES = {"exactitud": "Exactitud", "precision": "Precisión",
           "sensibilidad": "Sensibilidad", "especificidad": "Especificidad",
           "dice": "Dice", "f1": "F1"}

def resumir(lista):
    return {m: (float(np.mean([r[m] for r in lista])),
                float(np.std ([r[m] for r in lista]))) for m in CLAVES}

resumen       = resumir(resultados_pliegue)
resumen_rot   = resumir(resultados_pliegue_rot)
resumen_fijos = resumir(resultados_pliegue_fijos)

print("=" * 72)
print(f"Watershed — validación cruzada de {N_SPLITS} pliegues ")
print("=" * 72)
print(f"{'Métrica':<16}{'Test total':>14}")
print("-" * 72)
for m in CLAVES:
    print(f"{NOMBRES[m]:<16}{resumen[m][0]:>14.4f}")
print("-" * 72)







Watershed — validación cruzada de 5 pliegues 
Métrica             Test total
------------------------------------------------------------------------
Exactitud               0.8593
Sensibilidad            0.8715
Especificidad           0.9001
Precisión               0.8622
Dice                    0.7991
F1                      0.7739
------------------------------------------------------------------------
